# Part 1: Search API

## Text search

In [2]:
from rcsbapi.search import TextQuery

query = TextQuery(value="Hemoglobin")
results = list(query())

print(f"Found {len(results)} entries")
print(results[:10])

Found 8960 entries
['3GOU', '6IHX', '2PGH', '9JYU', '9RXG', '3PEL', '3PI9', '3PIA', '1FSX', '1QPW']


## Attribute search

In [16]:
from rcsbapi.search import AttributeQuery

query = AttributeQuery(
    attribute="rcsb_entity_source_organism.scientific_name",
    operator="exact_match",
    value="Homo sapiens"
)

results = list(query())
print(f"Found {len(results)} entries")
print(results[:10])

Found 79453 entries
['10AD', '10DC', '10DJ', '10FT', '10GH', '10GS', '10IJ', '10IK', '10JT', '10KR']


In [18]:
from rcsbapi.search import search_attributes as attrs
query = attrs.rcsb_entity_source_organism.scientific_name == "Homo sapiens"
results = list(query())
print(f"Found {len(results)} entries")
print(results[:10])

Found 79453 entries
['10AD', '10DC', '10DJ', '10FT', '10GH', '10GS', '10IJ', '10IK', '10JT', '10KR']


## Combining queries

In [20]:
from rcsbapi.search import TextQuery
from rcsbapi.search import search_attributes as attrs

q1 = TextQuery(value="Hemoglobin")
q2 = attrs.rcsb_entity_source_organism.scientific_name == "Homo sapiens"

query = q1 & q2
results = list(query())

print(f"Found {len(results)} entries")
print(results[:10])

Found 1633 entries
['1SHR', '1I3D', '1SI4', '1FDH', '1Y01', '1JEB', '1I3E', '1W0B', '1Z8U', '7QU4']


# Part 2: Data API

## Example 1: get the experimental method for one structure

In [4]:
from rcsbapi.data import DataQuery

query = DataQuery(
    input_type="entries",
    input_ids=["4HHB"],
    return_data_list=["exptl.method"]
)

result = query.exec()
result

{'data': {'entries': [{'rcsb_id': '4HHB',
    'exptl': [{'method': 'X-RAY DIFFRACTION'}]}]}}

In [8]:
result['data']['entries'][0]['rcsb_id']

'4HHB'

## Example 2: retrieve more than one field

In [23]:
from rcsbapi.data import DataQuery

query = DataQuery(
    input_type="entries",
    input_ids=["4HHB"],
    return_data_list=["rcsb_id", "struct.title", "exptl.method"]
)

result = query.exec()
result

{'data': {'entries': [{'rcsb_id': '4HHB',
    'struct': {'title': 'THE CRYSTAL STRUCTURE OF HUMAN DEOXYHAEMOGLOBIN AT 1.74 ANGSTROMS RESOLUTION'},
    'exptl': [{'method': 'X-RAY DIFFRACTION'}]}]}}

## Example 3: request data for multiple entries

In [9]:
from rcsbapi.data import DataQuery

query = DataQuery(
    input_type="entries",
    input_ids=["4HHB", "1CRN"],
    return_data_list=["rcsb_id", "struct.title"]
)

result = query.exec()
result

{'data': {'entries': [{'rcsb_id': '4HHB',
    'struct': {'title': 'THE CRYSTAL STRUCTURE OF HUMAN DEOXYHAEMOGLOBIN AT 1.74 ANGSTROMS RESOLUTION'}},
   {'rcsb_id': '1CRN',
    'struct': {'title': 'WATER STRUCTURE OF A HYDROPHOBIC PROTEIN AT ATOMIC RESOLUTION. PENTAGON RINGS OF WATER MOLECULES IN CRYSTALS OF CRAMBIN'}}]}}

In [14]:
result['data']['entries'][1]['rcsb_id']

'1CRN'

In [1]:
from rcsbapi.search import TextQuery
from rcsbapi.search import search_attributes as attrs
from rcsbapi.data import DataQuery


def unique_keep_order(items):
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


# Step 1: repeat Exercise 4
q1 = TextQuery(value="kinase")
q2 = attrs.rcsb_entity_source_organism.scientific_name == "Homo sapiens"
query = q1 & q2

pdb_ids = list(query())[:10]
print("10 PDB IDs:")
print(pdb_ids)


# Step 2: build the final dictionary
resulting_dictionary = {}

for pdb_id in pdb_ids:
    pdb_id = pdb_id.upper()

    # Get assembly IDs and polymer entity IDs for this entry
    entry_query = DataQuery(
        input_type="entries",
        input_ids=[pdb_id],
        return_data_list=[
            "rcsb_entry_container_identifiers.assembly_ids",
            "rcsb_entry_container_identifiers.polymer_entity_ids",
        ],
    )
    entry_result = entry_query.exec()
    entry_data = entry_result["data"]["entries"][0]

    assembly_ids = entry_data["rcsb_entry_container_identifiers"].get("assembly_ids", [])
    polymer_entity_ids = entry_data["rcsb_entry_container_identifiers"].get("polymer_entity_ids", [])

    # Build chain -> UniProt mapping
    chain_to_uniprot = {}

    for entity_id in polymer_entity_ids:
        polymer_entity_query = DataQuery(
            input_type="polymer_entities",
            input_ids=[f"{pdb_id}_{entity_id}"],
            return_data_list=[
                "rcsb_polymer_entity_container_identifiers.asym_ids",
                "rcsb_polymer_entity_container_identifiers.uniprot_ids",
            ],
        )
        polymer_entity_result = polymer_entity_query.exec()
        polymer_entity_data = polymer_entity_result["data"]["polymer_entities"][0]

        container = polymer_entity_data["rcsb_polymer_entity_container_identifiers"]
        asym_ids = container.get("asym_ids", [])
        uniprot_ids = container.get("uniprot_ids", [])

        if len(uniprot_ids) == 0:
            uniprot_value = None
        elif len(uniprot_ids) == 1:
            uniprot_value = uniprot_ids[0]
        else:
            uniprot_value = uniprot_ids

        for asym_id in asym_ids:
            chain_to_uniprot[asym_id] = uniprot_value

    # Build bas -> chain -> UniProt
    resulting_dictionary[pdb_id] = {}

    for assembly_id in assembly_ids:
        assembly_query = DataQuery(
            input_type="assemblies",
            input_ids=[f"{pdb_id}-{assembly_id}"],
            return_data_list=[
                "rcsb_assembly_container_identifiers.assembly_id",
                "pdbx_struct_assembly_gen.asym_id_list",
            ],
        )
        assembly_result = assembly_query.exec()
        assembly_data = assembly_result["data"]["assemblies"][0]

        assembly_gen = assembly_data.get("pdbx_struct_assembly_gen", [])

        asym_ids_in_assembly = []
        for item in assembly_gen:
            asym_ids_in_assembly.extend(item.get("asym_id_list", []))

        asym_ids_in_assembly = unique_keep_order(asym_ids_in_assembly)

        bas_key = f"bas_{assembly_id}"
        resulting_dictionary[pdb_id][bas_key] = {}

        for asym_id in asym_ids_in_assembly:
            if asym_id in chain_to_uniprot:
                resulting_dictionary[pdb_id][bas_key][f"chain {asym_id}"] = chain_to_uniprot[asym_id]

print(resulting_dictionary)

10 PDB IDs:
['1QK1', '1I0E', '4O75', '5J5T', '2CU1', '4UY9', '9P6A', '3DTC', '4IDT', '5K28']
{'1QK1': {'bas_1': {'chain A': 'P12532', 'chain B': 'P12532', 'chain C': 'P12532', 'chain D': 'P12532', 'chain E': 'P12532', 'chain F': 'P12532', 'chain G': 'P12532', 'chain H': 'P12532'}}, '1I0E': {'bas_1': {'chain A': 'P06732'}, 'bas_2': {'chain B': 'P06732'}, 'bas_3': {'chain C': 'P06732', 'chain D': 'P06732'}, 'bas_4': {'chain C': 'P06732', 'chain D': 'P06732'}}, '4O75': {'bas_1': {'chain A': 'O60885'}}, '5J5T': {'bas_1': {'chain A': 'Q8IVH8'}}, '2CU1': {'bas_1': {'chain A': 'Q9Y2U5'}}, '4UY9': {'bas_1': {'chain A': 'P80192', 'chain B': 'P80192'}}, '9P6A': {'bas_1': {'chain A': 'Q9Y2U5'}, 'bas_2': {'chain B': 'Q9Y2U5'}}, '3DTC': {'bas_1': {'chain A': 'P80192'}}, '4IDT': {'bas_1': {'chain A': 'Q99558', 'chain B': 'Q99558'}}, '5K28': {'bas_1': {'chain A': 'Q16584'}, 'bas_2': {'chain B': 'Q16584'}}}


In [2]:
resulting_dictionary

{'1QK1': {'bas_1': {'chain A': 'P12532',
   'chain B': 'P12532',
   'chain C': 'P12532',
   'chain D': 'P12532',
   'chain E': 'P12532',
   'chain F': 'P12532',
   'chain G': 'P12532',
   'chain H': 'P12532'}},
 '1I0E': {'bas_1': {'chain A': 'P06732'},
  'bas_2': {'chain B': 'P06732'},
  'bas_3': {'chain C': 'P06732', 'chain D': 'P06732'},
  'bas_4': {'chain C': 'P06732', 'chain D': 'P06732'}},
 '4O75': {'bas_1': {'chain A': 'O60885'}},
 '5J5T': {'bas_1': {'chain A': 'Q8IVH8'}},
 '2CU1': {'bas_1': {'chain A': 'Q9Y2U5'}},
 '4UY9': {'bas_1': {'chain A': 'P80192', 'chain B': 'P80192'}},
 '9P6A': {'bas_1': {'chain A': 'Q9Y2U5'}, 'bas_2': {'chain B': 'Q9Y2U5'}},
 '3DTC': {'bas_1': {'chain A': 'P80192'}},
 '4IDT': {'bas_1': {'chain A': 'Q99558', 'chain B': 'Q99558'}},
 '5K28': {'bas_1': {'chain A': 'Q16584'}, 'bas_2': {'chain B': 'Q16584'}}}